In [ ]:
import calendar
import json
import os
import pickle
import random
import re
import sys
from datetime import date
from typing import List

import dateutil
import graphistry
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
import seaborn as sns
from pyspark.sql import DataFrame, SparkSession

In [ ]:
GRAPHISTRY_KEY_ID = os.environ["GRAPHISTRY_KEY_ID"]
GRAPHISTRY_API_KEY = os.environ["GRAPHISTRY_API_KEY"]

In [ ]:
# Configuration for Graphistry
GRAPHISTRY_PARAMS = {
    "play": 500,
    "pointOpacity": 0.7,
    "edgeOpacity": 0.3,
    "edgeCurvature": 0.3,
    "showArrows": True,
    "gravity": 0.15,
    "showPointsOfInterestLabel": False,
    "labels": {
        "shortenLabels": False,
    },
}
FAVICON_URL = "https://graphlet.ai/assets/icons/favicon.ico"
LOGO = {"url": "https://graphlet.ai/assets/Branding/Graphlet%20AI.svg", "dimensions": {"maxWidth": 100, "maxHeight": 100}}

In [ ]:
# Initialize a SparkSession
spark: SparkSession = (
    SparkSession.builder.appName("Stack Overflow Pregel API")
    # Lets the Id:(Stack Overflow int) and id:(GraphFrames ULID) coexist
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints")

## Loading the Knowledge Graph

We used BAML and Gemini 2.5. Pro to extract the entities below and then PySpark to process the raw records into Parquet files and another script to create a node/edge list for the knowledge graph.

In [ ]:
!ls ../data/knowledge_graph_refined